In [1]:
# SparkSession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import datetime

spark = SparkSession.builder \
    .master("spark://spark:7077") \
    .appName("TestApp") \
    .config("spark.jars",
            "/opt/spark/jars-extra/hadoop-aws-3.3.4.jar,/opt/spark/jars-extra/aws-java-sdk-bundle-1.12.367.jar,/opt/spark/jars-extra/mssql-jdbc-12.6.0.jre11.jar") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "2613LApf.minio") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Hora:", datetime.datetime.now())

Hora: 2026-06-14 02:09:23.895511


In [12]:
# Se debe realizar pip install hvac en Terminal para poder usar Hashicorp Vault

In [14]:
# Crear Credenciales en Hashicorp Vault
import hvac

# Conexión a Vault vía Nginx
client = hvac.Client(
    url="http://vault.luispicado.com", 
    token="REDACTED_VAULT_TOKEN"
)

# Guardar credenciales de SQL Server BLL en el engine "secret"
client.secrets.kv.v2.create_or_update_secret(
    path="mssql",
    secret={
        "username": "sql_userbll",
        "password": "bll.1234!"
    },
    mount_point="secret"
)

{'request_id': 'd82256d7-72ca-9195-2f26-25c371cbbf09',
 'lease_id': '',
 'renewable': False,
 'lease_duration': 0,
 'data': {'created_time': '2026-06-14T02:08:14.978135593Z',
  'custom_metadata': None,
  'deletion_time': '',
  'destroyed': False,
  'version': 2},
 'wrap_info': None,
 'warnings': None,
 'auth': None,
 'mount_type': 'kv'}

In [2]:
# Leer el secreto de SQL Server BLL desde Hashicorp Vault
import hvac

client = hvac.Client(
    url="http://vault.luispicado.com",
    token="REDACTED_VAULT_TOKEN"
)

# Leer el secreto desde el engine "secret"
secret = client.secrets.kv.v2.read_secret_version(
    path="mssql",
    mount_point="secret",
    raise_on_deleted_version=True
)

print("Usuario:", secret["data"]["data"]["username"])
print("Password:", secret["data"]["data"]["password"])
print("Hora:", datetime.datetime.now())

Usuario: sql_userbll
Password: bll.1234!
Hora: 2026-06-14 02:09:26.513990


In [3]:
# Leer Tabla del SQL Server BLL usando el Secret desde Hashicorp Vault
import hvac

client = hvac.Client(
    url="http://vault.luispicado.com",
    token="REDACTED_VAULT_TOKEN"
)

# Leer el secreto desde el engine "secret"
secret = client.secrets.kv.v2.read_secret_version(
    path="mssql",
    mount_point="secret",
    raise_on_deleted_version=True
)

user = secret["data"]["data"]["username"]
pwd = secret["data"]["data"]["password"]

# Cadena de conexion
jdbc_url = (
    f"jdbc:sqlserver://msql.luispicado.com:51430;"
    f"databaseName=AdventureWorks2022;"
    f"user={user};password={pwd};"
    f"encrypt=true;trustServerCertificate=true"
)

df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Sales.SalesOrderHeader") \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .load()

df.show(10)
print("Hora:", datetime.datetime.now())

+------------+--------------+-------------------+-------------------+-------------------+------+---------------+----------------+-------------------+--------------+----------+-------------+-----------+---------------+---------------+------------+------------+----------------------+--------------+----------+---------+---------+----------+-------+--------------------+-------------------+
|SalesOrderID|RevisionNumber|          OrderDate|            DueDate|           ShipDate|Status|OnlineOrderFlag|SalesOrderNumber|PurchaseOrderNumber| AccountNumber|CustomerID|SalesPersonID|TerritoryID|BillToAddressID|ShipToAddressID|ShipMethodID|CreditCardID|CreditCardApprovalCode|CurrencyRateID|  SubTotal|   TaxAmt|  Freight|  TotalDue|Comment|             rowguid|       ModifiedDate|
+------------+--------------+-------------------+-------------------+-------------------+------+---------------+----------------+-------------------+--------------+----------+-------------+-----------+---------------+-----

In [ ]:
# Guardar en Minio
table_name = "SalesOrderHeader"

df.write \
  .mode("overwrite") \
  .parquet(f"s3a://bronce/Sales/{table_name}/")

In [ ]:
# Leer desde Minio
table_name = "SalesOrderHeader"

df_parquet = spark.read.parquet(f"s3a://bronce/Sales/{table_name}/")

df_parquet.show(10)